# Inspecting gadgets, checks, and readouts

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How does a gadget connect a lower-layer circuit and its measurements to logical blocks and syndromes?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Explain how a gadget's `in` and `out` boundary encodings map logical blocks to lower-layer qubits.
- Write and interpret modulo-2 parity `checks` and `readouts` across `in[b]`, `circuit.readouts[k]`, and `out[b]`.
- Inspect concrete `C4` gadget circuits and render timeline diagrams when a gadget uses `stim` format.

## Setting up

The cell below builds the baseline `C4` protocol and extracts its Layer 0 gadgets.

In [1]:
from _unit import make_baseline_protocol, show_stim_diagram

protocol = make_baseline_protocol()
gadgets = protocol.layers[0].gadgets

## Boundary encodings and the decoding surface

In general—independent of any particular code or circuit text format—a **gadget** (`qodec.Gadget`) is the per-instruction lowering rule between two adjacent layers of a `qodec`. While an instruction states *what* logical operation occurs, its gadget specifies:
1. **Boundary encodings (`inputs`, `outputs`):** how each incoming and outgoing encoded block maps onto target-layer qubit indices (leaving additional indices free for ancilla or flag qubits).
2. **A target-layer `circuit`:** a sequence of calls drawn from the next layer's instruction set.
3. **The decoding surface (`checks`, `readouts`):** linear parity equations over $\mathbb{F}_2$ (modulo-2 XOR) that relate boundary stabilizer/logical signs to the measurement bits emitted by the circuit:

<div style="text-align:center;"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 272" width="100%" style="max-width:100%;height:auto" class="qdk-ec-gadget" role="img" data-asset="tutorial_gadget_boundary.svg" aria-label="General anatomy of a qodec gadget connecting input and output block encodings across a target-layer circuit to parity checks and logical readouts."> <style> .qdk-ec-gadget { --qdk-fg: var(--vscode-editor-foreground, var(--jp-content-font-color1, currentColor)); --qdk-muted: color-mix(in srgb, var(--qdk-fg) 80%, transparent); --qdk-border: color-mix(in srgb, var(--qdk-fg) 30%, transparent); font-family: "Segoe UI", -apple-system, BlinkMacSystemFont, Roboto, Helvetica, Arial, sans-serif; } .qdk-ec-gadget .cluster-blue { fill: color-mix(in srgb, #1976d2 8%, transparent); stroke: color-mix(in srgb, #1976d2 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-gadget .cluster-teal { fill: color-mix(in srgb, #0d9488 8%, transparent); stroke: color-mix(in srgb, #0d9488 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-gadget .cluster-purple { fill: color-mix(in srgb, #7c3aed 8%, transparent); stroke: color-mix(in srgb, #7c3aed 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-gadget .cluster-neutral { fill: color-mix(in srgb, var(--qdk-fg) 4%, transparent); stroke: color-mix(in srgb, var(--qdk-fg) 32%, transparent); stroke-width: 1.8; } .qdk-ec-gadget .cluster-red { fill: color-mix(in srgb, #dc2626 9%, transparent); stroke: color-mix(in srgb, #dc2626 62%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-gadget .stage-blue { fill: color-mix(in srgb, #1976d2 18%, transparent); stroke: color-mix(in srgb, #1976d2 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-gadget .stage-teal { fill: color-mix(in srgb, #0d9488 18%, transparent); stroke: color-mix(in srgb, #0d9488 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-gadget .stage-purple { fill: color-mix(in srgb, #7c3aed 18%, transparent); stroke: color-mix(in srgb, #7c3aed 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-gadget .stage-red { fill: color-mix(in srgb, #dc2626 16%, transparent); stroke: color-mix(in srgb, #dc2626 78%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-gadget .loop-x { fill: color-mix(in srgb, #1976d2 20%, transparent); stroke: color-mix(in srgb, #1976d2 85%, var(--qdk-fg)); stroke-width: 2.2; } .qdk-ec-gadget .loop-z { fill: color-mix(in srgb, #0d9488 20%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2.2; stroke-dasharray: 6 3; } .qdk-ec-gadget .hdr-blue { fill: color-mix(in srgb, #1976d2 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-gadget .hdr-teal { fill: color-mix(in srgb, #0d9488 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-gadget .hdr-purple { fill: color-mix(in srgb, #7c3aed 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-gadget .hdr-neutral { fill: var(--qdk-fg); font-size: 12.2px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-gadget .hdr-red { fill: color-mix(in srgb, #dc2626 75%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-gadget .box-title { fill: var(--qdk-fg); font-size: 11.6px; font-weight: 700; } .qdk-ec-gadget .box-sub { fill: var(--qdk-muted); font-size: 10px; } .qdk-ec-gadget .code-tag { fill: var(--qdk-fg); font-family: Consolas, "Courier New", monospace; font-size: 9.4px; font-weight: 600; } .qdk-ec-gadget .badge-blue { fill: #1976d2; } .qdk-ec-gadget .badge-teal { fill: #0d9488; } .qdk-ec-gadget .badge-purple { fill: #7c3aed; } .qdk-ec-gadget .badge-red { fill: #dc2626; } .qdk-ec-gadget .badge-txt { fill: #ffffff; font-size: 9.6px; font-weight: 700; } .qdk-ec-gadget .conn { stroke: color-mix(in srgb, var(--qdk-fg) 68%, transparent); stroke-width: 2.2; fill: none; } .qdk-ec-gadget .wire { stroke: color-mix(in srgb, var(--qdk-fg) 48%, transparent); stroke-width: 1.5; fill: none; } .qdk-ec-gadget .gate-neutral { stroke: color-mix(in srgb, var(--qdk-fg) 78%, transparent); stroke-width: 1.8; fill: none; } .qdk-ec-gadget .gate-red { stroke: #dc2626; stroke-width: 2.3; fill: none; } .qdk-ec-gadget .gate-teal { stroke: #0d9488; stroke-width: 2.4; fill: none; } .qdk-ec-gadget .qubit-node { fill: var(--vscode-editor-background, #1e1e1e); stroke: color-mix(in srgb, var(--qdk-fg) 75%, transparent); stroke-width: 2; } .qdk-ec-gadget .anc-node { fill: color-mix(in srgb, #0d9488 22%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2; } </style> <defs> <marker id="gdg-arr" viewBox="0 0 10 10" refX="7.5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto-start-reverse"> <path d="M 0 1.5 L 8.5 5 L 0 8.5 z" fill="var(--qdk-fg)"/> </marker> </defs> <rect x="14" y="24" width="194" height="108" rx="8" class="stage-blue"/> <text x="111" y="48" text-anchor="middle" class="box-title">Input Encodings (in)</text> <text x="111" y="68" text-anchor="middle" class="box-sub">Maps input blocks to qubits</text> <text x="111" y="86" text-anchor="middle" class="box-sub">(empty on preparations)</text> <text x="111" y="112" text-anchor="middle" class="code-tag">in[b].stabilizers &#183; x, z</text> <path d="M 210 78 L 238 78" class="conn" marker-end="url(#gdg-arr)"/> <rect x="242" y="16" width="416" height="126" rx="10" class="cluster-teal"/> <text x="450" y="38" text-anchor="middle" class="hdr-teal">GADGET CIRCUIT (TARGET ISA)</text> <text x="450" y="60" text-anchor="middle" class="box-sub">Target-layer operations on block support and ancilla qubits</text> <text x="450" y="80" text-anchor="middle" class="box-sub">Resets, unitary gates, and measurements</text> <text x="450" y="106" text-anchor="middle" class="box-sub">Measurements record indexed classical bits:</text> <text x="450" y="126" text-anchor="middle" class="code-tag">circuit.readouts[0], circuit.readouts[1], ...</text> <path d="M 660 78 L 688 78" class="conn" marker-end="url(#gdg-arr)"/> <rect x="692" y="24" width="194" height="108" rx="8" class="stage-blue"/> <text x="789" y="48" text-anchor="middle" class="box-title">Output Encodings (out)</text> <text x="789" y="68" text-anchor="middle" class="box-sub">Maps output blocks to qubits</text> <text x="789" y="86" text-anchor="middle" class="box-sub">(empty on destructive readouts)</text> <text x="789" y="112" text-anchor="middle" class="code-tag">out[b].stabilizers &#183; x, z</text> <path d="M 355 142 L 355 159 L 242 159 L 242 174" class="conn" marker-end="url(#gdg-arr)"/> <path d="M 545 142 L 545 159 L 658 159 L 658 174" class="conn" marker-end="url(#gdg-arr)"/> <rect x="46" y="176" width="392" height="82" rx="8" class="cluster-blue"/> <text x="242" y="198" text-anchor="middle" class="hdr-blue">CHECKS (XOR PARITY = 0 IF FAULT-FREE)</text> <text x="242" y="218" text-anchor="middle" class="box-sub">Constraints relating boundary stabilizers &amp; measurement bits</text> <text x="242" y="240" text-anchor="middle" class="code-tag">XOR of listed references = 0 without faults</text> <rect x="462" y="176" width="392" height="82" rx="8" class="cluster-purple"/> <text x="658" y="198" text-anchor="middle" class="hdr-purple">READOUTS (OBSERVABLES THEN FLAGS)</text> <text x="658" y="218" text-anchor="middle" class="box-sub">Positional logical outcomes first, followed by named flags</text> <text x="658" y="240" text-anchor="middle" class="code-tag">XOR of incoming frames &amp; measurement bits</text> </svg></div>

<div style="text-align:center;">

*General gadget anatomy: boundary encodings (`in[b]`, `out[b]`) wrap a target-layer `circuit`, while modulo-2 `checks` and `readouts` expose syndrome invariants, observables, and flags.*

</div>

Why does `qodec` write parity equations across `in[b]`, `circuit.readouts[k]`, and `out[b]` instead of asserting that an ancilla measurement bit `circuit.readouts[k]` simply equals `0`? Because gadgets must compose across multi-round programs where prior gates, measurements, or software Pauli frames have flipped stabilizer or logical signs. Writing a check as a list of bit references asserts that their **modulo-2 sum is zero** on any fault-free execution:
- `in[b].stabilizers[i]` and `out[b].stabilizers[i]` reference the sign bit ($0$ for $+1$, $1$ for $-1$) of stabilizer `i` entering or leaving block `b`.
- `in[b].x[j]` and `in[b].z[j]` reference incoming logical **frame** bits—classical bookkeeping for known Pauli flips tracked in software rather than corrected with physical gates.
- `circuit.readouts[k]` references the $k$-th measurement bit emitted by `gadget.circuit`.

<div style="text-align:center;"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 940 286" width="100%" style="max-width:100%;height:auto" class="qdk-ec-parity" role="img" data-asset="tutorial_parity_references.svg" aria-label="How qodec expresses checks and readouts as modulo-2 XOR sums over input block signs, circuit measurement bits, and output block signs."> <style> .qdk-ec-parity { --qdk-fg: var(--vscode-editor-foreground, var(--jp-content-font-color1, currentColor)); --qdk-muted: color-mix(in srgb, var(--qdk-fg) 80%, transparent); --qdk-border: color-mix(in srgb, var(--qdk-fg) 30%, transparent); font-family: "Segoe UI", -apple-system, BlinkMacSystemFont, Roboto, Helvetica, Arial, sans-serif; } .qdk-ec-parity .cluster-blue { fill: color-mix(in srgb, #1976d2 8%, transparent); stroke: color-mix(in srgb, #1976d2 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-parity .cluster-teal { fill: color-mix(in srgb, #0d9488 8%, transparent); stroke: color-mix(in srgb, #0d9488 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-parity .cluster-purple { fill: color-mix(in srgb, #7c3aed 8%, transparent); stroke: color-mix(in srgb, #7c3aed 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-parity .cluster-neutral { fill: color-mix(in srgb, var(--qdk-fg) 4%, transparent); stroke: color-mix(in srgb, var(--qdk-fg) 32%, transparent); stroke-width: 1.8; } .qdk-ec-parity .cluster-red { fill: color-mix(in srgb, #dc2626 9%, transparent); stroke: color-mix(in srgb, #dc2626 62%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-parity .stage-blue { fill: color-mix(in srgb, #1976d2 18%, transparent); stroke: color-mix(in srgb, #1976d2 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-parity .stage-teal { fill: color-mix(in srgb, #0d9488 18%, transparent); stroke: color-mix(in srgb, #0d9488 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-parity .stage-purple { fill: color-mix(in srgb, #7c3aed 18%, transparent); stroke: color-mix(in srgb, #7c3aed 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-parity .stage-red { fill: color-mix(in srgb, #dc2626 16%, transparent); stroke: color-mix(in srgb, #dc2626 78%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-parity .loop-x { fill: color-mix(in srgb, #1976d2 20%, transparent); stroke: color-mix(in srgb, #1976d2 85%, var(--qdk-fg)); stroke-width: 2.2; } .qdk-ec-parity .loop-z { fill: color-mix(in srgb, #0d9488 20%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2.2; stroke-dasharray: 6 3; } .qdk-ec-parity .hdr-blue { fill: color-mix(in srgb, #1976d2 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-parity .hdr-teal { fill: color-mix(in srgb, #0d9488 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-parity .hdr-purple { fill: color-mix(in srgb, #7c3aed 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-parity .hdr-neutral { fill: var(--qdk-fg); font-size: 12.2px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-parity .hdr-red { fill: color-mix(in srgb, #dc2626 75%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-parity .box-title { fill: var(--qdk-fg); font-size: 11.6px; font-weight: 700; } .qdk-ec-parity .box-sub { fill: var(--qdk-muted); font-size: 10px; } .qdk-ec-parity .code-tag { fill: var(--qdk-fg); font-family: Consolas, "Courier New", monospace; font-size: 9.4px; font-weight: 600; } .qdk-ec-parity .badge-blue { fill: #1976d2; } .qdk-ec-parity .badge-teal { fill: #0d9488; } .qdk-ec-parity .badge-purple { fill: #7c3aed; } .qdk-ec-parity .badge-red { fill: #dc2626; } .qdk-ec-parity .badge-txt { fill: #ffffff; font-size: 9.6px; font-weight: 700; } .qdk-ec-parity .conn { stroke: color-mix(in srgb, var(--qdk-fg) 68%, transparent); stroke-width: 2.2; fill: none; } .qdk-ec-parity .wire { stroke: color-mix(in srgb, var(--qdk-fg) 48%, transparent); stroke-width: 1.5; fill: none; } .qdk-ec-parity .gate-neutral { stroke: color-mix(in srgb, var(--qdk-fg) 78%, transparent); stroke-width: 1.8; fill: none; } .qdk-ec-parity .gate-red { stroke: #dc2626; stroke-width: 2.3; fill: none; } .qdk-ec-parity .gate-teal { stroke: #0d9488; stroke-width: 2.4; fill: none; } .qdk-ec-parity .qubit-node { fill: var(--vscode-editor-background, #1e1e1e); stroke: color-mix(in srgb, var(--qdk-fg) 75%, transparent); stroke-width: 2; } .qdk-ec-parity .anc-node { fill: color-mix(in srgb, #0d9488 22%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2; } </style> <defs> <marker id="par-arr" viewBox="0 0 10 10" refX="7.5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto-start-reverse"> <path d="M 0 1.5 L 8.5 5 L 0 8.5 z" fill="var(--qdk-fg)"/> </marker> </defs> <rect x="14" y="14" width="336" height="258" rx="10" class="cluster-neutral"/> <text x="182" y="36" text-anchor="middle" class="hdr-neutral">THREE BIT NAMESPACES (0 = +1, 1 = &#8722;1)</text> <rect x="28" y="48" width="308" height="64" rx="7" class="stage-blue"/> <text x="42" y="67" class="box-title">1. Input Block State &#8212; in[b]</text> <text x="42" y="85" class="code-tag">in[0].stabilizers[i]</text> <text x="192" y="85" class="box-sub">incoming stabilizer sign</text> <text x="42" y="102" class="code-tag">in[0].z[j], in[0].x[j]</text> <text x="192" y="102" class="box-sub">incoming Pauli frame</text> <rect x="28" y="122" width="308" height="64" rx="7" class="stage-teal"/> <text x="42" y="141" class="box-title">2. Circuit Measurements &#8212; circuit</text> <text x="42" y="160" class="code-tag">circuit.readouts[k]</text> <text x="42" y="177" class="box-sub">k-th classical measurement bit in circuit order</text> <rect x="28" y="196" width="308" height="64" rx="7" class="stage-blue"/> <text x="42" y="215" class="box-title">3. Output Block State &#8212; out[b]</text> <text x="42" y="233" class="code-tag">out[0].stabilizers[i]</text> <text x="196" y="233" class="box-sub">outgoing stabilizer sign</text> <text x="42" y="250" class="code-tag">out[0].z[j], out[0].x[j]</text> <text x="196" y="250" class="box-sub">outgoing Pauli frame</text> <path d="M 354 143 L 386 143" class="conn" marker-end="url(#par-arr)"/> <rect x="390" y="14" width="536" height="258" rx="10" class="cluster-neutral"/> <text x="658" y="36" text-anchor="middle" class="hdr-neutral">LISTS MEAN MODULO-2 XOR &#8212; C4 EXAMPLES</text> <rect x="406" y="48" width="504" height="64" rx="7" class="cluster-blue"/> <text x="420" y="67" class="box-title">Ancilla Check (syndrome): ancilla bit matches incoming sign</text> <text x="420" y="87" class="code-tag">[&quot;in[0].stabilizers[0]&quot;, &quot;circuit.readouts[0]&quot;]</text> <text x="420" y="104" class="box-sub">Asserts s0_in &#8853; r0 = 0 &#8594; chained across rounds without assuming r0 = 0</text> <rect x="406" y="122" width="504" height="64" rx="7" class="cluster-teal"/> <text x="420" y="141" class="box-title">Boundary Conservation Check (cx_all / syndrome): zero measurements</text> <text x="420" y="161" class="code-tag">[&quot;in[0].stabilizers[0]&quot;, &quot;out[0].stabilizers[0]&quot;]</text> <text x="420" y="178" class="box-sub">Asserts s0_in &#8853; s0_out = 0 &#8594; propagates stabilizer signs to the next gadget</text> <rect x="406" y="196" width="504" height="64" rx="7" class="cluster-purple"/> <text x="420" y="215" class="box-title">Observable Readout (measure_z_all, z[0] = Z_0 Z_2): value expression</text> <text x="420" y="235" class="code-tag">[&quot;in[0].z[0]&quot;, &quot;circuit.readouts[0]&quot;, &quot;circuit.readouts[2]&quot;]</text> <text x="420" y="252" class="box-sub">Defines outcome = z0_in &#8853; r0 &#8853; r2 &#8594; folds incoming software frame into Z_0 Z_2</text> </svg></div>

<div style="text-align:center;">

*Three bit namespaces (`in[b]`, `circuit.readouts[k]`, `out[b]`) combine via modulo-2 XOR to express ancilla checks, unitary stabilizer conservation, and frame-corrected logical observable readouts.*

</div>

## Inspecting C4 measurement, unitary, and syndrome gadgets

With the general model in place, let's inspect how three `C4` gadgets instantiate these fields:
1. `measure_z_all` destructively measures qubits `0..3`: its four physical bits yield **two observable readouts** ($\bar{Z}_0 = Z_0 Z_2$ and $\bar{Z}_1 = Z_0 Z_1$, each XORed with `in[0].z[j]`) plus **one parity check** ($Z_0 Z_1 Z_2 Z_3$ XORed with `in[0].stabilizers[1]`).
2. `cx_all` applies four transversal CNOTs across qubits `0..7` with **zero measurements**, yet declares **four checks** relating input stabilizer signs on both blocks (`in[0]`, `in[1]`) to output signs (`out[0]`, `out[1]`).
3. `syndrome` allocates ancillas `4` and `5` to measure $X_0 X_1 X_2 X_3$ and $Z_0 Z_1 Z_2 Z_3$, emitting two ancilla-readout checks plus two input-to-output stabilizer conservation checks:

In [2]:
for name in ["measure_z_all", "cx_all", "syndrome"]:
    g = gadgets[name]
    print(f"=== {name} ===")
    print("  in      :", list(g.inputs))
    print("  out     :", list(g.outputs))
    print("  checks  :", list(g.checks))
    print("  readouts:", list(g.readouts))

=== measure_z_all ===
  in      : [Encoding(code="C4")]
  out     : []
  checks  : [(Reference('circuit.readouts[0]'), Reference('circuit.readouts[1]'), Reference('circuit.readouts[2]'), Reference('circuit.readouts[3]'), Reference('in[0].stabilizers[1]'))]
  readouts: [Readout(position=0, name=None, is_flag=False, equation=['in[0].z[0]', 'circuit.readouts[0]', 'circuit.readouts[2]']), Readout(position=1, name=None, is_flag=False, equation=['in[0].z[1]', 'circuit.readouts[0]', 'circuit.readouts[1]'])]
=== cx_all ===
  in      : [Encoding(code="C4"), Encoding(code="C4")]
  out     : [Encoding(code="C4"), Encoding(code="C4")]
  checks  : [(Reference('out[0].stabilizers[0]'), Reference('in[0].stabilizers[0]'), Reference('in[1].stabilizers[0]')), (Reference('out[0].stabilizers[1]'), Reference('in[0].stabilizers[1]')), (Reference('out[1].stabilizers[0]'), Reference('in[1].stabilizers[0]')), (Reference('out[1].stabilizers[1]'), Reference('in[0].stabilizers[1]'), Reference('in[1].stabilizers[1

### Inspect the syndrome circuit

`qodec` supports multiple circuit text formats (`stim`, OpenQASM 3, and custom instruction sets, as we will use in Unit 08). When a gadget stores its circuit in `format="stim"`, as `build_qodec` does here, we can render a timeline diagram directly from `gadget.circuit.source`:

In [3]:
print("Circuit format:", gadgets["syndrome"].circuit.format)
print(gadgets["syndrome"].circuit.source)
show_stim_diagram(gadgets["syndrome"].circuit.source)

Circuit format: stim
R 4
H 4
CX 4 0
CX 4 1
CX 4 2
CX 4 3
H 4
R 5
H 5
CZ 5 0
CZ 5 1
CZ 5 2
CZ 5 3
H 5
M 4 5



<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does measure_z_all produce two readouts and one check from four physical qubit measurements?</summary>

<div style="padding:0.1em 1em;">

`measure_z_all` reports the two logical $Z$ observables (`z[0] = Z_0 Z_2` and `z[1] = Z_0 Z_1`) as two `readouts`. Meanwhile, the weight-four stabilizer `Z_0 Z_1 Z_2 Z_3` requires the XOR of all four measurement bits to match `in[0].stabilizers[1]`, giving one parity `check`.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why do syndrome checks XOR incoming stabilizer signs with ancilla measurement bits instead of requiring M 4 = 0?</summary>

<div style="padding:0.1em 1em;">

If an earlier operation flipped the incoming block into the $-1$ eigenspace of a stabilizer (`in[0].stabilizers[i] = 1`), a fault-free ancilla measurement records `1`. Requiring `in[0].stabilizers[i] ^ circuit.readouts[i] == 0` expresses the invariant relative to the incoming stabilizer sign so gadgets compose across rounds.

</div>

</details></div>

## Exercise: Author parity readouts and checks for measure_x_all

The `C4` `measure_x_all` gadget applies `H 0 1 2 3` followed by `M 0 1 2 3`, so `circuit.readouts[0..3]` record single-qubit $X$ eigenvalues on physical qubits `0..3`. Recall that on `C4`, `stabilizers[0] = X_0 X_1 X_2 X_3`, `x[0] = X_0 X_1`, and `x[1] = X_0 X_2`. Starting from a cleared `mx` gadget (`mx.readouts = []`, `mx.checks = []`), populate `mx.readouts` (two logical $X$ observable parities in order `x[0]`, `x[1]`) and `mx.checks` (the four-qubit $X$ stabilizer parity), including the corresponding `in[0]` frame/stabilizer terms, and return `mx`.

In [ ]:
import qodec
from _unit import exercise, make_baseline_protocol


@exercise
def author_measure_x_equations() -> qodec.Gadget:
    mx = make_baseline_protocol().layers[0].gadgets["measure_x_all"]
    mx.readouts = []  # Populate observable parities for x[0] and x[1]
    mx.checks = []  # Populate the X_0 X_1 X_2 X_3 stabilizer check
    return mx

**Hint**

Mirror `measure_z_all`, replacing `z` with `x` and `stabilizers[1]` with `stabilizers[0]`: `x[0] = X_0 X_1` pairs `in[0].x[0]` with `circuit.readouts[0]` and `circuit.readouts[1]`; `x[1] = X_0 X_2` pairs `in[0].x[1]` with `circuit.readouts[0]` and `circuit.readouts[2]`; and the check XORs `in[0].stabilizers[0]` with all four `circuit.readouts[0..3]`.

In [ ]:
import qodec
from _unit import exercise, make_baseline_protocol


@exercise
def author_measure_x_equations() -> qodec.Gadget:
    mx = make_baseline_protocol().layers[0].gadgets["measure_x_all"]
    mx.readouts = [
        ["in[0].x[0]", "circuit.readouts[0]", "circuit.readouts[1]"],
        ["in[0].x[1]", "circuit.readouts[0]", "circuit.readouts[2]"],
    ]
    mx.checks = [
        [
            "in[0].stabilizers[0]",
            "circuit.readouts[0]",
            "circuit.readouts[1]",
            "circuit.readouts[2]",
            "circuit.readouts[3]",
        ],
    ]
    return mx

After `H 0 1 2 3`, `M 0 1 2 3` measures each data qubit in the $X$ basis. Including `in[0].x[0]` and `in[0].x[1]` in `mx.readouts` incorporates any tracked incoming logical $X$-sign flips, while XORing `in[0].stabilizers[0]` with all four physical bits checks the weight-four $X_0 X_1 X_2 X_3$ stabilizer (notice $Z_0 Z_1 Z_2 Z_3$ anticommutes with $X$-basis measurement and therefore yields no check).

## Further reading

- [Gadget concepts and boundary encodings ↗](https://github.com/microsoft/qdk-ec/blob/main/qodec/docs/concepts/gadget.md)
- [Using qodec with QDK, deq, and Stim ↗](https://github.com/microsoft/qdk-ec/blob/main/qodec/docs/ecosystem.md)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../02-instruction-sets/instruction_sets.workbook.ipynb" title="Previous unit: Building a Baseline Qodec and Instruction Sets" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Building a Baseline Qodec and Instruction Sets</span></span></a><a href="../04-validate-and-audit/validate_and_audit.workbook.ipynb" title="Next unit: Validating and Auditing a Protocol" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Validating and Auditing a Protocol</span></span><span aria-hidden="true">&#8250;</span></a></div>